# P22 live local notebook

Run top to bottom to see environment, approval state, dataset size, donor splitting, train-only transforms, six-model five-seed benchmark, gated-model interventions, stress sizing, resource use, and final gate.

This notebook generates synthetic arrays in memory. It does not download, read, or analyze real disease data. Real-data work remains blocked until Professor Fang approves dataset and protocol.

In [1]:
import json
import platform
import resource
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch

import p22
from p22.data import make_donor_split
from p22.data.transforms import fit_train_only
from p22.eval.faithfulness import INTERVENTIONS, intervention_table, run_all_interventions
from p22.models import GatedFusionModel
from p22.runs.config import load_run_config
from p22.testing import make_synthetic_multimodal
from p22.training import train_model
from p22.training.seed_runner import run_all_seeds, summarize_across_seeds

warnings.filterwarnings('ignore', message='y_pred contains classes not in y_true')
CONFIG_PATH = p22.REPO_ROOT / 'configs' / 'scale_pilot.json'
OUTPUT_DIR = p22.REPO_ROOT / 'reports' / 'generated' / 'live_local'
RUN_STRESS = True
config = load_run_config(CONFIG_PATH)

def peak_rss_mb():
    value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return value / (1024 * 1024) if sys.platform == 'darwin' else value / 1024

def matrix_mib(n_cells, n_features):
    return n_cells * n_features * 4 / (1024 * 1024)

print('repository:', p22.REPO_ROOT)
print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('torch:', torch.__version__)
print('device: cpu')
print('data mode:', config.data_mode)
print('approval blocked:', p22.approval_blocked())
print('initial process peak RSS MB:', round(peak_rss_mb(), 1))

repository: /Users/anuragdani/Github/niw-eb1a/P22
python: 3.11.15
platform: macOS-26.5.2-arm64-arm-64bit
torch: 2.8.0
device: cpu
data mode: synthetic
approval blocked: True
initial process peak RSS MB: 372.8


## 1. Dataset size and memory estimate

Medium tier is the full model run. Stress tier is data-path only. Raw matrix size uses float32 and excludes model, transform, metadata, and Jupyter overhead.

In [2]:
medium_cells = config.dataset['n_donors'] * config.dataset['cells_per_donor']
medium_raw_mib = matrix_mib(medium_cells, config.dataset['n_features_a']) + matrix_mib(medium_cells, config.dataset['n_features_b'])
size_table = pd.DataFrame([
    {'tier': 'medium full pipeline', 'donors': config.dataset['n_donors'], 'cells': medium_cells, 'features/view': config.dataset['n_features_a'], 'raw float32 matrices MiB': round(medium_raw_mib, 1)},
    {'tier': 'stress data path', 'donors': 128, 'cells': 128 * 250, 'features/view': 1000, 'raw float32 matrices MiB': round(matrix_mib(128 * 250, 1000) * 2, 1)},
])
size_table

,tier,donors,cells,features/view,raw float32 matrices MiB
0,medium full pipeline,64,6400,256,12.5
1,stress data path,128,32000,1000,244.1


## 2. Generate data, split donors, fit transforms

The same donor-first and train-only contracts used by the benchmark run here visibly.

In [3]:
medium_data = make_synthetic_multimodal(**config.dataset, seed=0)
medium_split = make_donor_split(medium_data.donor_ids, labels=medium_data.labels, seed=0)
holdout_index = np.concatenate([medium_split.val_index, medium_split.test_index])
train_ids = medium_data.cell_ids[medium_split.train_index]
holdout_ids = medium_data.cell_ids[holdout_index]
fitted = {name: fit_train_only(matrix, medium_data.cell_ids, train_ids, holdout_ids, seed=0) for name, matrix in {'view_a': medium_data.view_a, 'view_b': medium_data.view_b}.items()}
transformed = {name: item.transform(getattr(medium_data, name)) for name, item in fitted.items()}
print('shapes:', {name: matrix.shape for name, matrix in transformed.items()})
print('donors per split:', medium_split.summary['realized_donor_counts'])
print('cells per split:', medium_split.summary['realized_cell_counts'])
print('train-only:', {name: item.metadata['fit_scope'] for name, item in fitted.items()})

shapes: {'view_a': (6400, 256), 'view_b': (6400, 256)}
donors per split: {'train': 38, 'val': 13, 'test': 13}
cells per split: {'train': 3800, 'val': 1300, 'test': 1300}
train-only: {'view_a': 'train cells only', 'view_b': 'train cells only'}


## 3. Full medium benchmark

Runs six baselines across five seeds. Test data is read once per model and donor intervals are reported.

In [4]:
benchmark_started = time.perf_counter()
benchmark_runs = run_all_seeds(config, seeds=config.seeds, n_replicates=config.bootstrap['n_replicates'])
benchmark_seconds = time.perf_counter() - benchmark_started
benchmark_summary = summarize_across_seeds(benchmark_runs)
benchmark_table = pd.DataFrame([{'model': name, 'accuracy mean': values['accuracy']['mean'], 'accuracy sd': values['accuracy']['std'], 'accuracy interval': values['accuracy']['interval'], 'seeds': values['accuracy']['n_seeds']} for name, values in benchmark_summary.items()]).sort_values('model')
print('runtime seconds:', round(benchmark_seconds, 3))
print('process peak RSS MB:', round(peak_rss_mb(), 1))
benchmark_table

runtime seconds: 37.659
process peak RSS MB: 583.3


,model,accuracy mean,accuracy sd,accuracy interval,seeds
0,concat_fusion,0.887538,0.034788,"[0.8443439438775717, 0.9307329791993513]",5
1,gated_fusion,0.887231,0.031438,"[0.8481954282577091, 0.9262661102038292]",5
2,logistic_regression_view_a,0.888000,0.029436,"[0.8514509773638744, 0.9245490226361258]",5
3,logistic_regression_view_b,0.876308,0.033368,"[0.8348756893349595, 0.9177396952804252]",5
4,mlp_view_a,0.889538,0.029410,"[0.8530206523114797, 0.9260562707654431]",5
5,mlp_view_b,0.881385,0.027485,"[0.8472570611179522, 0.9155121696512785]",5


## 4. Gated fusion and faithfulness interventions

One gated model is trained on the visible medium dataset. Seven held-out interventions measure prediction changes under stated manipulations. These are synthetic intervention results, not causal biology.

In [5]:
parts = {part: {name: matrix[index] for name, matrix in transformed.items()} for part, index in (('train', medium_split.train_index), ('val', medium_split.val_index), ('test', medium_split.test_index))}
torch.manual_seed(0)
gated = GatedFusionModel(n_features_a=transformed['view_a'].shape[1], n_features_b=transformed['view_b'].shape[1], n_classes=medium_data.n_classes, embed_dim=config.training['embedding_dim'], hidden_dim=config.training['hidden_dim'])
trained = train_model(gated, parts['train'], medium_data.labels[medium_split.train_index], parts['val'], medium_data.labels[medium_split.val_index], max_epochs=config.training['max_epochs'], batch_size=config.training['batch_size'], learning_rate=config.training['learning_rate'], patience=config.training['patience'], seed=0)
effects = run_all_interventions(trained.model, parts['test'], medium_data.labels[medium_split.test_index], medium_data.donor_ids[medium_split.test_index], train_views=parts['train'], seed=0)
intervention_df = pd.DataFrame(intervention_table(effects)).round(4)
print('best validation score:', round(trained.best_val_score, 4))
print('interventions:', len(effects))
intervention_df

best validation score: 0.8904
interventions: 7


,intervention,target_view,flip_rate,confidence_drop,balanced_accuracy_before,balanced_accuracy_after,metric_drop,routing_shift,status
0,clamp_view_a,view_a,0.2077,0.1074,0.8569,0.7334,0.1234,0.0936,measured
1,clamp_view_b,view_b,0.1646,0.0722,0.8569,0.8053,0.0516,0.0600,measured
2,permute_view_a_within_donor,view_a,0.3269,0.0470,0.8569,0.6318,0.2250,0.0817,measured
3,permute_view_b_within_donor,view_b,0.3108,0.0463,0.8569,0.6554,0.2014,0.0681,measured
4,ablate_view_a_embedding,view_a,0.2092,0.1062,0.8569,0.7337,0.1232,0.0971,measured
5,ablate_view_b_embedding,view_b,0.1669,0.0720,0.8569,0.8022,0.0546,0.0619,measured
6,fixed_uniform_route,-,0.0192,0.0103,0.8569,0.8567,0.0001,0.0746,measured


## 5. Optional larger stress tier

Set `RUN_STRESS = False` near the top on machines with limited RAM. Stress tier generates 32,000 cells and 1,000 features per view, then checks donor splitting and train-only transforms without full model training.

In [6]:
stress_result = {'status': 'SKIPPED'}
if RUN_STRESS:
    stress_spec = {'n_donors': 128, 'cells_per_donor': 250, 'n_features_a': 1000, 'n_features_b': 1000, 'n_classes': 3, 'class_signal': 0.9, 'donor_nuisance': 0.7, 'noise': 1.0}
    stress_started = time.perf_counter()
    stress_data = make_synthetic_multimodal(**stress_spec, seed=91)
    stress_split = make_donor_split(stress_data.donor_ids, labels=stress_data.labels, seed=91)
    stress_holdout = np.concatenate([stress_split.val_index, stress_split.test_index])
    stress_train_ids = stress_data.cell_ids[stress_split.train_index]
    stress_holdout_ids = stress_data.cell_ids[stress_holdout]
    stress_a = fit_train_only(stress_data.view_a, stress_data.cell_ids, stress_train_ids, stress_holdout_ids)
    stress_b = fit_train_only(stress_data.view_b, stress_data.cell_ids, stress_train_ids, stress_holdout_ids)
    stress_donor_sets = [set(stress_split.donors(name)) for name in ('train', 'val', 'test')]
    stress_overlap = sum(len(left & right) for index, left in enumerate(stress_donor_sets) for right in stress_donor_sets[index + 1:])
    stress_result = {'status': 'PASS', 'cells': stress_data.n_cells, 'donors': stress_data.n_donors, 'features_per_view': 1000, 'seconds': round(time.perf_counter() - stress_started, 3), 'peak_rss_mb': round(peak_rss_mb(), 1), 'donor_overlap': stress_overlap, 'transformed_shapes': [list(stress_a.transform(stress_data.view_a[:2]).shape), list(stress_b.transform(stress_data.view_b[:2]).shape)]}
print(json.dumps(stress_result, indent=2))

{
  "status": "PASS",
  "cells": 32000,
  "donors": 128,
  "features_per_view": 1000,
  "seconds": 0.932,
  "peak_rss_mb": 1636.6,
  "donor_overlap": 0,
  "transformed_shapes": [
    [
      2,
      1000
    ],
    [
      2,
      1000
    ]
  ]
}


## 6. Final local gate and saved summary

Generated summary is written under ignored `reports/generated/live_local/`. Source code and raw data are not changed.

In [7]:
checks = {'synthetic mode': config.data_mode == p22.SYNTHETIC, 'approval remains blocked': p22.approval_blocked(), 'medium has five seeds': len(benchmark_runs) == 5, 'six models run each seed': all(len(run.outcomes) == 6 for run in benchmark_runs), 'donors never cross splits': all(run.split_summary['donor_overlap'] == 0 for run in benchmark_runs), 'seven interventions measured': len(effects) == len(INTERVENTIONS), 'stress passed when enabled': not RUN_STRESS or stress_result['status'] == 'PASS', 'stress donor overlap is zero': not RUN_STRESS or stress_result['donor_overlap'] == 0}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(checks.values()), 'live local gate failed'
summary = {'data_mode': config.data_mode, 'approval_blocked': p22.approval_blocked(), 'medium_config': config.to_dict(), 'medium_seconds': benchmark_seconds, 'medium_peak_rss_mb': peak_rss_mb(), 'benchmark': benchmark_summary, 'interventions': intervention_df.to_dict(orient='records'), 'stress': stress_result, 'limitations': list(config.limitations)}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / 'P22_live_local_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, default=str) + '\n', encoding='utf-8')
print('summary:', summary_path.relative_to(p22.REPO_ROOT))
print('LIVE LOCAL gate: PASS')

PASS: synthetic mode
PASS: approval remains blocked
PASS: medium has five seeds
PASS: six models run each seed
PASS: donors never cross splits
PASS: seven interventions measured
PASS: stress passed when enabled
PASS: stress donor overlap is zero
summary: reports/generated/live_local/P22_live_local_summary.json
LIVE LOCAL gate: PASS


## Interpretation

This live run demonstrates reproducible synthetic engineering behavior. It does not establish real-cohort adequacy, disease biology, cross-cohort generalization, or causal interpretation. Real-data execution requires Professor Fang approval, metadata preflight, frozen estimand, donor-held-out split, and independent validation.